# Flow Matching Neural Process (FlowNP) — Experiment Guide

This notebook documents the full FlowNP training pipeline for irregular spatial datasets.
It covers the **configuration files**, **data loading**, **model architecture**,
**training loop**, **optimisation algorithm**, **checkpointing**, and
**evaluation metrics** — everything needed to reproduce or extend experiments.

---

## Table of Contents

1. [Project Structure](#1.-Project-Structure)
2. [Background: Neural Processes Meet Flow Matching](#2.-Background)
3. [Configuration Files (YAML)](#3.-Configuration-Files)
4. [Data Pipeline](#4.-Data-Pipeline)
5. [Model Architecture](#5.-Model-Architecture)
6. [Training Loop & Optimisation](#6.-Training-Loop-&-Optimisation)
7. [Evaluation Metrics](#7.-Evaluation-Metrics)
8. [Checkpointing & Resuming](#8.-Checkpointing-&-Resuming)
9. [Running Experiments from the CLI](#9.-Running-Experiments)
10. [Live Demo: Mini Training Run](#10.-Live-Demo)

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
os.chdir(os.path.dirname(os.path.abspath("experiment_guide.ipynb")) 
         if os.path.exists("experiment_guide.ipynb") 
         else "/home/srahimis/SPL-Baseline-Neurips2026/flownp_spl")

import numpy as np
import torch
import yaml
from pathlib import Path
from torch.utils.data import DataLoader

from data import SpatialDataset, spatial_collate
from model import FlowNP, fourier_encode, _build_mlp

print(f"PyTorch {torch.__version__}")
print(f"Working directory: {os.getcwd()}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Project Structure

```
flownp_spl/
├── configs/
│   ├── weather.yaml        # Hyperparameters for PeakWeather dataset
│   └── gdp.yaml            # Hyperparameters for GDP ocean-drifter dataset
├── data.py                 # SpatialDataset + spatial_collate (variable-length support)
├── model.py                # FlowNP model (Fourier encoding, Transformer, velocity head)
├── train.py                # End-to-end training script (CLI entry point)
└── experiment_guide.ipynb  # This notebook
```

| File | Responsibility |
|------|----------------|
| **`data.py`** | Loads `.npz` files, creates random context/target splits per sample, pads variable-length sequences in a batch, and produces boolean masks. |
| **`model.py`** | Defines the `FlowNP` nn.Module — Fourier positional encoding, token embedding MLP, Transformer encoder, velocity prediction head. Implements the CFM training loss (`forward`) and stochastic Euler sampling (`predict`). |
| **`train.py`** | Parses a YAML config, builds data loaders / model / optimiser / scheduler, runs the epoch loop, evaluates periodically, and saves checkpoints. |
| **`configs/*.yaml`** | All tuneable hyperparameters live here — data paths, model architecture, training schedule, and evaluation settings. No code changes are needed to switch datasets. |

## 2. Background

### 2.1 Neural Processes (NPs)

Neural Processes are a family of meta-learning models that learn to map a
**context set** of observations $\{(x_c^{(i)}, y_c^{(i)})\}_{i=1}^{N_c}$ to a
predictive distribution over function values at **target locations**
$\{x_t^{(j)}\}_{j=1}^{N_t}$.  They combine the flexibility of neural networks
with the principled uncertainty quantification of stochastic processes.

The key idea: instead of fitting a single function, NPs learn a *distribution
over functions* from data, enabling them to provide calibrated uncertainty
estimates and adapt to new tasks at test time with a single forward pass.

### 2.2 Flow Matching for Generative Modelling

Flow Matching (Lipman et al., 2023) provides a **simulation-free** training
objective for continuous normalising flows.  Instead of solving an ODE during
training, it trains a velocity field $v_\theta(x, t)$ to match the time
derivative of a simple probability path that interpolates between a source
distribution (e.g. Gaussian noise at $t=0$) and the data distribution
(at $t=1$):

$$x_t = t \cdot x_1 + (1 - t) \cdot x_0, \qquad x_0 \sim \mathcal{N}(0, I),\; x_1 \sim p_{\text{data}}$$

The **conditional flow matching (CFM) loss** is simply:

$$\mathcal{L}_{\text{CFM}} = \mathbb{E}_{t, x_0, x_1}\left[\|v_\theta(x_t, t) - (x_1 - x_0)\|^2\right]$$

This is an MSE regression — no ODE solving, no divergence estimation, no
adversarial training.  At inference, samples are generated by integrating
the learned velocity from $t=0$ to $t=1$.

### 2.3 Flow Matching Neural Processes (FlowNP)

FlowNP (NeurIPS 2025) unifies these two ideas.  Instead of predicting
a Gaussian mean and variance at target locations (like CNP/ANP), it uses
**conditional flow matching** as the generative mechanism:

1. **Context encoding**: context points $(x_c, y_c)$ are Fourier-encoded
   and processed jointly with target tokens through a Transformer encoder.
2. **Velocity prediction**: a head on top of the Transformer predicts the
   velocity field $v_\theta(y_t, t \mid \text{context})$ for the target
   values at flow-time $t$.
3. **Training**: the CFM-MSE loss — same as standard flow matching, but
   conditioned on the context set.
4. **Inference**: Euler integration of the velocity field from Gaussian noise
   produces function samples at target locations.

## 3. Configuration Files

All experiment hyperparameters are stored in YAML files under `configs/`.
The training script reads the YAML and uses it to construct the data loaders,
model, optimiser, and scheduler — **no code changes are needed to switch
between datasets or to tune hyperparameters**.

Let's load and inspect both configs side by side.

In [ ]:
with open("configs/weather.yaml") as f:
    weather_cfg = yaml.safe_load(f)
with open("configs/gdp.yaml") as f:
    gdp_cfg = yaml.safe_load(f)

print("=" * 70)
print("PeakWeather config (configs/weather.yaml)")
print("=" * 70)
for k, v in weather_cfg.items():
    print(f"  {k:25s} = {v}")

print()
print("=" * 70)
print("GDP config (configs/gdp.yaml)")
print("=" * 70)
for k, v in gdp_cfg.items():
    print(f"  {k:25s} = {v}")

### 3.1 Config field reference

The YAML files are organised into four logical groups.  Every field is
explained below.

#### Data fields

| Field | Type | Description |
|-------|------|-------------|
| `data_path` | `str` | Absolute path to the `.npz` file containing `x_train`, `pos_train`, `x_test`, `pos_test` (object arrays of variable-length samples). |
| `dataset_name` | `str` | Short identifier used for checkpoint directory naming (e.g. `"peakweather"`, `"gdp"`). |
| `x_dim` | `int` | Spatial coordinate dimensionality. PeakWeather uses 2D Swiss grid coordinates; GDP uses 3D unit-sphere XYZ. |
| `y_dim` | `int` | Feature / observation dimensionality. PeakWeather has 4 channels (temperature, pressure, humidity, wind speed); GDP has 3 (eastward velocity, northward velocity, sea-surface temperature). |
| `context_frac_range` | `[float, float]` | During training, a random fraction in this range determines how many points become context vs. target. E.g. `[0.1, 0.5]` means 10–50 % of the points are context. |
| `eval_context_frac` | `float` | Fixed context fraction used during evaluation (deterministic split for reproducibility). |
| `num_workers` | `int` | Number of `DataLoader` worker processes. |

#### Model architecture fields

| Field | Type | Description |
|-------|------|-------------|
| `dim_posenc` | `int` | Number of Fourier features per input coordinate dimension. Each spatial dimension + the flow-time dimension is encoded with `dim_posenc` sinusoidal features. Must be even. |
| `d_model` | `int` | Hidden dimension of the Transformer encoder. |
| `emb_depth` | `int` | Depth (number of linear layers) of the token embedding MLP that maps raw tokens to `d_model`-dimensional embeddings. |
| `dim_feedforward` | `int` | Hidden dimension of the feed-forward layers inside each Transformer encoder layer, and also the intermediate dimension of the velocity prediction head. |
| `nhead` | `int` | Number of attention heads in each Transformer encoder layer. Must divide `d_model` evenly. |
| `dropout` | `float` | Dropout rate applied inside the Transformer layers. |
| `num_layers` | `int` | Number of stacked Transformer encoder layers. |
| `timesteps` | `int` | Number of Euler integration steps used during sampling/prediction. More steps = better approximation but slower inference. |

#### Training fields

| Field | Type | Description |
|-------|------|-------------|
| `batch_size` | `int` | Mini-batch size for both training and evaluation data loaders. |
| `learning_rate` | `float` | Initial learning rate for the Adam optimiser. |
| `weight_decay` | `float` | L2 regularisation coefficient for Adam. |
| `epochs` | `int` | Total number of training epochs. |

#### Evaluation & checkpointing fields

| Field | Type | Description |
|-------|------|-------------|
| `eval_interval` | `int` | Run evaluation every this many epochs (and always at the final epoch). |
| `eval_num_samples` | `int` | Number of stochastic sample trajectories used in the final sample-based prediction evaluation. |

### 3.2 Differences between the two dataset configs

The two configs share the same model architecture and training schedule.
The only differences reflect the nature of the underlying data:

| Parameter | PeakWeather | GDP |
|-----------|:-----------:|:---:|
| `x_dim` | **2** (Swiss Easting/Northing) | **3** (unit-sphere x/y/z) |
| `y_dim` | **4** (temp, pressure, humidity, wind) | **3** (ve, vn, sst) |
| Points per sample | ~128 (fixed stations) | 64–220 (variable drifters) |

Everything else — `d_model`, `nhead`, `num_layers`, learning rate, etc. —
is identical, ensuring a fair comparison under the same capacity budget.

## 4. Data Pipeline

The data pipeline is implemented in `data.py` and consists of two components:

1. **`SpatialDataset`** — a PyTorch `Dataset` that loads the `.npz`, picks a
   random context/target split for each sample, and returns variable-length
   tensors.
2. **`spatial_collate`** — a custom collation function that pads variable-length
   context and target sets to the batch maximum and creates boolean masks.

### 4.1 The `.npz` format

Each dataset file contains NumPy **object arrays** (because samples have
different numbers of observation points):

```
x_train:   shape (N_train,),  each element (n_i, y_dim)   — features
pos_train: shape (N_train,),  each element (n_i, x_dim)   — coordinates
x_test:    shape (N_test,),   each element (n_i, y_dim)
pos_test:  shape (N_test,),   each element (n_i, x_dim)
```

Let's inspect the actual data.

In [ ]:
# Inspect both datasets
for name, path in [("PeakWeather", weather_cfg["data_path"]),
                   ("GDP",         gdp_cfg["data_path"])]:
    data = np.load(path, allow_pickle=True)
    print(f"{'=' * 50}")
    print(f" {name}  ({path.split('/')[-1]})")
    print(f"{'=' * 50}")
    print(f"  Keys:            {list(data.keys())}")
    print(f"  Train samples:   {data['x_train'].shape[0]}")
    print(f"  Test  samples:   {data['x_test'].shape[0]}")
    print(f"  Feature names:   {list(data['parameters'])}")
    print(f"  Coord system:    {data['coord_system']}")
    
    # Show shape variability
    train_sizes = [data["x_train"][i].shape[0] for i in range(min(200, len(data["x_train"])))]
    print(f"  Points/sample:   min={min(train_sizes)}, max={max(train_sizes)}, "
          f"mean={np.mean(train_sizes):.1f}")
    print(f"  Sample 0 shapes: features={data['x_train'][0].shape}, "
          f"positions={data['pos_train'][0].shape}")
    print()

### 4.2 `SpatialDataset` — random context/target splits

Each call to `__getitem__` randomly partitions the observation points into a
**context set** (what the model sees) and a **target set** (what the model
must predict).

The context fraction is drawn uniformly from `context_frac_range` (e.g. 10–50%).
During evaluation, `deterministic=True` seeds the RNG with the sample index
so every evaluation pass sees the exact same splits.

In [ ]:
# Build a small dataset and look at one sample
data = np.load(weather_cfg["data_path"], allow_pickle=True)
ds = SpatialDataset(data["x_train"][:50], data["pos_train"][:50],
                    context_frac_range=(0.1, 0.5))

sample = ds[0]
print("Keys returned by SpatialDataset.__getitem__:")
for k, v in sample.items():
    print(f"  {k:15s}  shape={str(v.shape):15s}  dtype={v.dtype}")

print(f"\nOriginal sample had {data['pos_train'][0].shape[0]} points total")
print(f"  -> {sample['x_context'].shape[0]} context + "
      f"{sample['x_target'].shape[0]} target")

### 4.3 `spatial_collate` — padding and masking

Because different samples in a batch may have different numbers of context
and target points, we cannot simply `torch.stack` them.  The custom
`spatial_collate` function:

1. Finds the maximum context/target length in the batch.
2. Zero-pads all samples to that maximum length.
3. Creates boolean **masks** (`True` for real points, `False` for padding).

These masks are passed to both the Transformer (via `src_key_padding_mask`)
and the loss function (to exclude padded positions from the gradient).

In [ ]:
# Show collated batch structure
loader = DataLoader(ds, batch_size=4, collate_fn=spatial_collate, drop_last=True)
batch = next(iter(loader))

print("Collated batch tensors:")
print(f"{'key':20s} {'shape':20s} {'dtype':15s} notes")
print("-" * 70)
for k, v in batch.items():
    notes = ""
    if "mask" in k:
        notes = f"  (True count per sample: {v.sum(dim=1).tolist()})"
    print(f"{k:20s} {str(list(v.shape)):20s} {str(v.dtype):15s}{notes}")

print("\nThe masks show that different samples have different real lengths.")
print("Padded positions are zeroed out and excluded from loss computation.")

## 5. Model Architecture

The `FlowNP` model in `model.py` has three main components:

```
Input tokens                          Output
─────────────                         ──────
 (posenc ‖ features)                  velocity v(y_t, t)
       │                                  ▲
       ▼                                  │
┌─────────────┐    ┌─────────────────┐    │
│  Embedding  │───▶│   Transformer   │───▶│  Velocity Head
│    MLP      │    │    Encoder      │    │  (Linear→ReLU→Linear)
│  (4 layers) │    │  (6 layers,     │    │
│             │    │   4 heads)      │    │
└─────────────┘    └─────────────────┘    │
                                          ▼
                                   (B, N_target, dim_y)
```

### 5.1 Fourier Positional Encoding

Spatial coordinates and the flow-time scalar are encoded using sinusoidal
features before being fed to the model.  For each coordinate dimension,
the encoding uses frequencies $\omega_k = \pi \cdot 2^{k-2}$ for
$k = 0, 1, \ldots, \frac{n_\text{freq}}{2}-1$:

$$\text{FourierEncode}(x) = \bigl[\sin(\omega_0 x), \cos(\omega_0 x), \sin(\omega_1 x), \cos(\omega_1 x), \ldots\bigr]$$

A coordinate vector $x \in \mathbb{R}^D$ produces an encoding of size
$D \times n_\text{freq}$.  With `dim_posenc=80`, each of the
`(dim_x + 1)` coordinates (spatial dims + flow-time) gets 80 features,
giving a total of `(dim_x + 1) * 80` positional features per token.

In [ ]:
# Demonstrate Fourier encoding on a simple 2D coordinate
demo_x = torch.tensor([[[0.5, 0.3]]])  # (1, 1, 2)  — one 2D point
encoded = fourier_encode(demo_x, n_freq=80)
print(f"Input shape:   {demo_x.shape}  (1 batch, 1 point, 2 coords)")
print(f"Encoded shape: {encoded.shape}  (1 batch, 1 point, 2*80 = 160 features)")
print(f"Encoding range: [{encoded.min():.3f}, {encoded.max():.3f}]")

### 5.2 Token construction and the velocity field

The model processes two types of tokens, concatenated into a single sequence:

| Token type | Positional input | Feature input | Purpose |
|------------|:----------------:|:-------------:|---------|
| **Context** | `FourierEncode(xc ‖ 1)` | `yc` | Observed data. Flow-time is fixed at `t=1` (fully "arrived"). |
| **Target**  | `FourierEncode(xt ‖ t)` | `yt` (noisy) | Locations where we want predictions. The noisy values `yt` are the current state along the flow. |

Each token is the concatenation `[positional_encoding ‖ features]`, giving
dimension `(dim_x + 1) * dim_posenc + dim_y`.

The **embedding MLP** maps this to `d_model` dimensions.  The **Transformer
encoder** processes all tokens with full (bidirectional) attention — context
tokens can attend to targets and vice versa.  After encoding, only the
**target-position outputs** are passed through the **velocity head** to
predict $v_\theta(y_t, t \mid \text{context})$.

### 5.3 Padding mask propagation

For variable-length batches, a `src_key_padding_mask` is constructed from
the context and target masks:

```python
pad_mask = ~cat([context_mask, target_mask], dim=1)   # True = ignore
```

This ensures the Transformer ignores padded tokens in its attention computation.

In [ ]:
# Instantiate the model and inspect its structure
model = FlowNP(dim_x=2, dim_y=4, dim_posenc=80, d_model=64, emb_depth=4,
               dim_feedforward=128, nhead=4, dropout=0.0, num_layers=6,
               timesteps=100)

print(f"Total parameters: {model.num_params:,} ({model.num_params/1e6:.2f}M)\n")

# Token dimension calculation
token_dim = (2 + 1) * 80 + 4  # (dim_x + 1) * dim_posenc + dim_y
print(f"Token dimension:  (dim_x + 1) * dim_posenc + dim_y")
print(f"                = ({2} + 1) * {80} + {4} = {token_dim}")
print()

# Show sub-modules
print("Model sub-modules:")
for name, module in model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"  {name:20s}  {n_params:>8,} params    {type(module).__name__}")

## 6. Training Loop & Optimisation

### 6.1 The Conditional Flow Matching (CFM) loss

The training objective is the **CFM-MSE** loss.  For each mini-batch:

1. **Sample noise**: $y_0 \sim \mathcal{N}(0, I)$, same shape as `y_target`.
2. **Sample time**: $t \sim \text{Uniform}(0, 1)$, independently per target point.
3. **Interpolate**: $y_t = t \cdot y_{\text{target}} + (1 - t) \cdot y_0$ — the noisy intermediate state.
4. **Predict velocity**: $v_\theta = \text{FlowNP}(x_c, y_c, x_t, y_t, t)$.
5. **Compute loss**: $\mathcal{L} = \| v_\theta - (y_{\text{target}} - y_0) \|^2$, averaged over valid target points.

The loss is **masked** to exclude padded target positions:

```python
sq_err = (v_pred - v_true).pow(2).mean(dim=-1)       # per-point MSE
n_valid = tar_mask.float().sum(dim=1).clamp(min=1)    # valid count per sample
loss = ((sq_err * tar_mask.float()).sum(1) / n_valid).mean()
```

This ensures that padded positions contribute zero gradient regardless of
the random values in the padding region.

In [ ]:
# Demonstrate a single training forward pass step-by-step
model.train()

# Use the batch we loaded earlier
xc, yc = batch["x_context"], batch["y_context"]
xt, yt = batch["x_target"],  batch["y_target"]
ctx_mask, tar_mask = batch["context_mask"], batch["target_mask"]

B, Nt, Dy = yt.shape
print(f"Batch: B={B}, N_target={Nt}, dim_y={Dy}")
print()

# Step 1: sample noise
y0 = torch.randn_like(yt)
print(f"Step 1 — Noise y0:       shape={y0.shape}")

# Step 2: sample random time
t = torch.rand(B, Nt, 1)
print(f"Step 2 — Time t:         shape={t.shape}   range=[{t.min():.3f}, {t.max():.3f}]")

# Step 3: linear interpolation
yt_noisy = t * yt + (1.0 - t) * y0
print(f"Step 3 — Interpolated:   shape={yt_noisy.shape}")

# Step 4: predict velocity through the model
v_pred = model._velocity(xc, yc, xt, yt_noisy, t, ctx_mask=ctx_mask, tar_mask=tar_mask)
print(f"Step 4 — Velocity pred:  shape={v_pred.shape}")

# Step 5: target velocity and masked loss
v_true = yt - y0
sq_err = (v_pred - v_true).pow(2).mean(dim=-1)
n_valid = tar_mask.float().sum(dim=1).clamp(min=1)
loss = ((sq_err * tar_mask.float()).sum(dim=1) / n_valid).mean()
print(f"Step 5 — CFM-MSE loss:   {loss.item():.6f}")
print(f"         (valid targets per sample: {n_valid.int().tolist()})")

### 6.2 Optimiser: Adam

We use the **Adam** optimiser (Kingma & Ba, 2015) with the following settings:

| Parameter | Value | Rationale |
|-----------|:-----:|-----------|
| Learning rate | `5e-4` | Standard for Transformer models on moderate-sized data. |
| Weight decay | `0.0` | No explicit L2 regularisation (dropout is also 0 — the datasets are large enough and overfitting is not the primary concern). |
| Betas | `(0.9, 0.999)` | PyTorch defaults — well-suited for flow matching objectives. |

Adam adapts the learning rate per-parameter using exponential moving averages
of the gradient (first moment) and squared gradient (second moment), which
helps with the varying gradient magnitudes across the embedding MLP,
Transformer layers, and velocity head.

### 6.3 Learning rate scheduler: Cosine Annealing

The learning rate follows a **cosine annealing** schedule
(`CosineAnnealingLR`) that decays from the initial value to ~0 over the
full training run:

$$\eta_t = \frac{\eta_0}{2}\left(1 + \cos\left(\frac{t \cdot \pi}{T_{\max}}\right)\right)$$

where $T_{\max}$ = `epochs`.  The scheduler is stepped **once per epoch**
(after each `train_epoch` call).

This schedule provides a warm start (high learning rate in early epochs for
fast convergence) and a gradual cooldown (low learning rate in late epochs
for fine-grained optimisation).

In [ ]:
# Visualise the cosine annealing schedule
import matplotlib
matplotlib.use("Agg")  # non-interactive backend
import matplotlib.pyplot as plt

epochs = 100
lr_init = 5e-4
lrs = [lr_init / 2 * (1 + np.cos(np.pi * e / epochs)) for e in range(epochs)]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(1, epochs + 1), lrs, linewidth=2, color="#2563eb")
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate")
ax.set_title("CosineAnnealingLR Schedule (T_max = 100 epochs)")
ax.set_xlim(1, epochs)
ax.set_ylim(0, lr_init * 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig("lr_schedule.png", dpi=120, bbox_inches="tight")
plt.show()
print("Schedule saved to lr_schedule.png")

### 6.4 The epoch-level training loop

The training loop in `train.py` follows this pseudocode:

```
best_loss ← ∞
for epoch = 1 to epochs:
    ┌─ train_epoch ─────────────────────────────────────────┐
    │  model.train()                                        │
    │  for each mini-batch from train_loader:               │
    │      move batch to device                             │
    │      out = model.forward(xc, yc, xt, yt, masks)      │
    │      optimizer.zero_grad()                            │
    │      out["loss"].backward()                           │
    │      optimizer.step()                                 │
    │      accumulate loss for reporting                    │
    │  return average epoch loss                            │
    └───────────────────────────────────────────────────────┘
    scheduler.step()                          # decay LR

    if epoch % eval_interval == 0:
        ┌─ evaluate_loss ──────────────────────────────────┐
        │  model.eval()   (no grad)                        │
        │  for each mini-batch from test_loader:           │
        │      out = model.forward(xc, yc, xt, yt, masks) │
        │      accumulate test CFM loss                    │
        │  return average test loss                        │
        └──────────────────────────────────────────────────┘
        if test_loss < best_loss:
            save best_model.pt
            best_loss ← test_loss

    save latest.pt   (every epoch)

# After all epochs:
run evaluate_prediction (slow, sample-based MSE)
```

Key design decisions:
- **Evaluation uses the same CFM loss** as training (fast — no ODE solving).
- **Sample-based prediction MSE** is only computed at the end (slow — requires
  `timesteps` forward passes per sample, times `num_samples` trajectories).
- **Two checkpoints** are saved: `best_model.pt` (best test loss) and
  `latest.pt` (most recent, for resuming).

## 7. Evaluation Metrics

The pipeline provides two complementary evaluation strategies:

### 7.1 Fast evaluation: CFM loss on test set (`evaluate_loss`)

This runs the **same conditional flow matching loss** on the held-out test set.
It is fast because it only requires a single forward pass per batch (no ODE
integration).  It measures how well the velocity field is learned.

- Called every `eval_interval` epochs during training.
- Used for best-model selection.

### 7.2 Sample-based evaluation: Prediction MSE (`evaluate_prediction`)

This generates actual **function samples** by Euler-integrating the learned
velocity field from Gaussian noise, then compares the predicted mean against
ground truth.  It measures end-to-end prediction quality.

For each test batch:
1. Start from $y_0 \sim \mathcal{N}(0, I)$.
2. Run `timesteps` Euler steps with the learned velocity + noise injection.
3. Repeat `num_samples` times and average.
4. Compute per-point MSE against `y_target`, masked for valid points.

- Only called once at the end of training (much slower).
- Reports the **MSE per valid target point**, averaged over the full test set.

The Euler update at each step uses a **stochastic scheme** from the FlowNP paper:

$$y_{t+\Delta t} = y_t + \left(\alpha(t) \cdot v_\theta(y_t, t) + \sigma(t) \cdot \epsilon\right) \cdot \Delta t$$

where $\alpha(t) = 1 + t(1-t)$, $\sigma(t) = 0.2\sqrt{t(1-t)}$, and
$\epsilon \sim \mathcal{N}(0, I)$.  The noise injection encourages sample
diversity; the time-dependent coefficients stabilise the integration near
$t=0$ and $t=1$.

In [ ]:
# Visualise the Euler step coefficients α(t) and σ(t)
t_vals = np.linspace(0, 1, 200)
alpha_vals = 1.0 + t_vals * (1.0 - t_vals)
sigma_vals = 0.2 * np.sqrt(np.maximum(t_vals * (1.0 - t_vals), 0.0))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(t_vals, alpha_vals, linewidth=2, color="#2563eb")
axes[0].set_xlabel("Flow time t")
axes[0].set_ylabel("α(t)")
axes[0].set_title("Velocity scaling α(t) = 1 + t(1-t)")
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_vals, sigma_vals, linewidth=2, color="#dc2626")
axes[1].set_xlabel("Flow time t")
axes[1].set_ylabel("σ(t)")
axes[1].set_title("Noise injection σ(t) = 0.2·√(t(1-t))")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.savefig("euler_coefficients.png", dpi=120, bbox_inches="tight")
plt.show()
print("Both coefficients peak at t=0.5 and vanish at the boundaries.")

## 8. Checkpointing & Resuming

### 8.1 What is saved

The training script saves **two checkpoint files** to the save directory
(default: `checkpoints/<dataset_name>/`):

| File | When saved | Contents |
|------|-----------|----------|
| `best_model.pt` | Whenever test loss improves | `epoch`, `model` state dict, `optimizer` state dict, `test_loss`, full `config` dict |
| `latest.pt` | Every epoch | `epoch`, `model` state dict, `optimizer` state dict, full `config` dict |

Both files are standard PyTorch checkpoint dicts saved with `torch.save`.
The **config is embedded** in every checkpoint, so you always know which
hyperparameters produced a given model.

### 8.2 Checkpoint structure

In [ ]:
# Show checkpoint structure from the test run (if it exists)
ckpt_path = Path("/tmp/flownp_test/best_model.pt")
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    print(f"Checkpoint: {ckpt_path}")
    print(f"  Top-level keys: {list(ckpt.keys())}")
    print(f"  Epoch:          {ckpt['epoch']}")
    print(f"  Test loss:      {ckpt.get('test_loss', 'N/A')}")
    print(f"  Model keys:     {len(ckpt['model'])} parameter tensors")
    print(f"  Optimizer keys: {list(ckpt['optimizer'].keys())}")
    print(f"  Config saved:   {list(ckpt['config'].keys())}")
else:
    print("(No test checkpoint found — run the Live Demo below to generate one)")

### 8.3 Loading a checkpoint for inference or resuming

To load a trained model for inference:

```python
import torch
from model import FlowNP

ckpt = torch.load("checkpoints/peakweather/best_model.pt", map_location="cpu")
cfg = ckpt["config"]

model = FlowNP(
    dim_x=cfg["x_dim"], dim_y=cfg["y_dim"],
    dim_posenc=cfg["dim_posenc"], d_model=cfg["d_model"],
    emb_depth=cfg["emb_depth"], dim_feedforward=cfg["dim_feedforward"],
    nhead=cfg["nhead"], num_layers=cfg["num_layers"],
    timesteps=cfg["timesteps"],
)
model.load_state_dict(ckpt["model"])
model.eval()

# Now use model.predict(xc, yc, xt, num_samples=50) for inference
```

To resume training, additionally restore the optimiser:

```python
optimizer = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])
optimizer.load_state_dict(ckpt["optimizer"])
start_epoch = ckpt["epoch"] + 1
```

> **Note**: The current `train.py` does not have a `--resume` flag built in,
> but the checkpoint format supports it — you just need to load the state
> dicts and adjust the epoch range.

## 9. Running Experiments

### 9.1 Command-line interface

The `train.py` script accepts three arguments:

| Argument | Required | Default | Description |
|----------|:--------:|---------|-------------|
| `--config` | Yes | — | Path to YAML config file. |
| `--device` | No | `cuda:0` | PyTorch device string (`cuda:0`, `cuda:1`, `cpu`). |
| `--save_dir` | No | `checkpoints/<dataset_name>/` | Override checkpoint save directory. |

### 9.2 Example commands

**Train on PeakWeather (GPU 0):**
```bash
cd flownp_spl
python train.py --config configs/weather.yaml --device cuda:0
```

**Train on GDP (GPU 1):**
```bash
cd flownp_spl
python train.py --config configs/gdp.yaml --device cuda:1
```

**Train both in parallel (different GPUs):**
```bash
cd flownp_spl
python train.py --config configs/weather.yaml --device cuda:0 &
python train.py --config configs/gdp.yaml     --device cuda:1 &
wait
```

**Custom save directory:**
```bash
python train.py --config configs/weather.yaml --save_dir runs/weather_v2
```

### 9.3 Expected output

Each epoch prints a summary line.  At evaluation epochs you also see the test loss:

```
Epoch   1/100  train_loss=0.828681  test_loss=0.475226  lr=2.50e-04  time=76.0s
  ↑ New best test loss: 0.475226
Epoch   2/100  train_loss=0.457146  lr=2.50e-04  time=75.5s
...
```

At the end of training, the slower sample-based evaluation runs:

```
Running sample-based evaluation (this is slower) …
Final prediction MSE per point: 0.167424
Best  CFM test loss:            0.425684
Checkpoints saved to checkpoints/peakweather
```

## 10. Live Demo: Mini Training Run

Below we run a **2-epoch training loop** directly in the notebook to verify
the full pipeline works end-to-end.  We use reduced settings for speed:
- `batch_size=8`, `num_workers=0`, `timesteps=5` (for fast prediction)
- Only 2 epochs with evaluation at each

In [ ]:
import time as _time
from train import train_epoch, evaluate_loss, evaluate_prediction, _to_device

# Use weather config with reduced settings for the demo
cfg = weather_cfg.copy()
device = torch.device("cpu")

# Load data
data = np.load(cfg["data_path"], allow_pickle=True)
train_ds = SpatialDataset(data["x_train"], data["pos_train"],
                          context_frac_range=tuple(cfg["context_frac_range"]))
test_ds  = SpatialDataset(data["x_test"], data["pos_test"],
                          context_frac_range=(cfg["eval_context_frac"],
                                              cfg["eval_context_frac"]),
                          deterministic=True)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,
                          num_workers=0, collate_fn=spatial_collate, drop_last=True)
test_loader  = DataLoader(test_ds, batch_size=8, shuffle=False,
                          num_workers=0, collate_fn=spatial_collate)

# Build model
demo_model = FlowNP(
    dim_x=cfg["x_dim"], dim_y=cfg["y_dim"],
    dim_posenc=cfg["dim_posenc"], d_model=cfg["d_model"],
    emb_depth=cfg["emb_depth"], dim_feedforward=cfg["dim_feedforward"],
    nhead=cfg["nhead"], dropout=cfg.get("dropout", 0.0),
    num_layers=cfg["num_layers"], timesteps=5,  # reduced for speed
).to(device)

optimizer = torch.optim.Adam(demo_model.parameters(), lr=cfg["learning_rate"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=2)

print(f"Model: {demo_model.num_params:,} parameters")
print(f"Train: {len(train_ds)} samples | Test: {len(test_ds)} samples")
print(f"{'=' * 60}")

# Train 2 epochs
history = {"epoch": [], "train_loss": [], "test_loss": [], "lr": []}

for epoch in range(1, 3):
    t0 = _time.time()
    train_loss = train_epoch(demo_model, train_loader, optimizer, device, report_freq=500)
    scheduler.step()
    test_loss = evaluate_loss(demo_model, test_loader, device)
    elapsed = _time.time() - t0

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["test_loss"].append(test_loss)
    history["lr"].append(scheduler.get_last_lr()[0])

    print(f"Epoch {epoch}/2  train_loss={train_loss:.6f}  test_loss={test_loss:.6f}  "
          f"lr={history['lr'][-1]:.2e}  time={elapsed:.1f}s")

print(f"\n{'=' * 60}")
print("Training complete — loss decreases as expected.")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(history["epoch"], history["train_loss"], "o-", color="#2563eb", label="Train")
axes[0].plot(history["epoch"], history["test_loss"],  "o-", color="#dc2626", label="Test")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("CFM Loss")
axes[0].set_title("Training & Test Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["epoch"], history["lr"], "o-", color="#16a34a")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("LR Schedule (2-epoch demo)")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.savefig("demo_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Run a quick prediction and show shapes
demo_model.eval()
test_batch = next(iter(test_loader))
test_batch = _to_device(test_batch, device)

mean, std = demo_model.predict(
    test_batch["x_context"], test_batch["y_context"],
    test_batch["x_target"], num_samples=5,
    ctx_mask=test_batch["context_mask"],
    tar_mask=test_batch["target_mask"],
)

print("Prediction results:")
print(f"  mean shape: {mean.shape}  — (batch, max_targets, y_dim)")
print(f"  std  shape: {std.shape}")
print(f"  mean range: [{mean.min():.3f}, {mean.max():.3f}]")
print(f"  std  range: [{std.min():.3f}, {std.max():.3f}]")

# Per-point MSE (masked)
mask = test_batch["target_mask"].float()
mse = ((mean - test_batch["y_target"]).pow(2).mean(dim=-1) * mask).sum() / mask.sum()
print(f"\n  Prediction MSE (this batch): {mse.item():.4f}")

## Summary

This notebook has covered every component of the FlowNP training pipeline:

| Component | Location | Section |
|-----------|----------|---------|
| YAML configs | `configs/weather.yaml`, `configs/gdp.yaml` | [Section 3](#3.-Configuration-Files) |
| Data loading & masking | `data.py` | [Section 4](#4.-Data-Pipeline) |
| Model architecture | `model.py` | [Section 5](#5.-Model-Architecture) |
| CFM loss function | `model.py → FlowNP.forward()` | [Section 6](#6.-Training-Loop-&-Optimisation) |
| Adam optimiser + cosine LR | `train.py` | [Section 6](#6.-Training-Loop-&-Optimisation) |
| Fast & slow evaluation | `train.py → evaluate_loss / evaluate_prediction` | [Section 7](#7.-Evaluation-Metrics) |
| Checkpointing | `train.py` (best + latest) | [Section 8](#8.-Checkpointing-&-Resuming) |
| CLI commands | `train.py --config ... --device ...` | [Section 9](#9.-Running-Experiments) |

**To run a full experiment**, simply execute:

```bash
cd flownp_spl
python train.py --config configs/weather.yaml --device cuda:0
```

All hyperparameters are in the YAML — no code changes needed.